# Lab 3: Cleaning I (Missing, Wrong, Duplicated)

**DSA 405 · Week 3**

| | |
|---|---|
| **In class** | Friday, Sep 4 |
| **A3 due** | Thursday, Sep 10, 11:59 PM |
| **File** | `wolfpack_dining_raw.csv` |
| **Time** | ~25 min in class, ~55 min at home |

## Overview

Lab 2 profiled the dining file and flagged columns not to trust. This Lab acts on
those findings and records every action taken. Each cleaning decision discards
information; the cleaning log states which, with counts.

The technical core is one idea in three forms: a value can be present, look plausible,
and still be wrong.

In [2]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

pandas 2.2.3


---
# Part 1: Explore (in class)

## Task 1.1: The sentinel census, and the two zeros

A **sentinel** is a special value written inside a data column to signal "no real value here," instead of leaving the cell empty.

Read everything as strings first (Lab 2 showed why). Then count what the `score` and
`seats` columns contain:

In [3]:
dining = load("wolfpack_dining_raw.csv", dtype=str)

print("--- score ---")
print(dining.score.str.strip().value_counts(dropna=False).head(8))
print()
print("--- seats ---")
print(dining.seats.str.strip().value_counts(dropna=False).head(8))

--- score ---
score
100.0    14
89.8     11
92.0      9
NaN       8
0         7
88.5      7
87.8*     6
96.3      6
Name: count, dtype: int64

--- seats ---
seats
8       23
2       20
6       20
7       19
1       15
0       14
-999    13
4       12
Name: count, dtype: int64


Two findings:

- `seats` contains **13** values of `-999` and **14** zeros
- `score` contains **7** zeros

The digit `0` appears in both columns. In one it is a legitimate value; in the other it
is impossible and must mean "not recorded." Deciding which is which, with evidence, is
Checkpoint question 1. Consider what each column measures: a food truck genuinely seats
nobody.

## Task 1.2: Type repair

`score` does not convert cleanly: some values carry whitespace, some a trailing `*`
(the file's way of marking revised scores), some the word `unknown`. Repair in steps
and check each step:

In [4]:
s = dining.score.str.strip().str.rstrip("*")
score = pd.to_numeric(s, errors="coerce")

print("failed to convert (now NaN):", score.isna().sum())
print("min:", score.min(), "| max:", score.max(), "| mean:", round(score.mean(), 1))

failed to convert (now NaN): 17
min: 0.0 | max: 100.0 | mean: 90.6


Note the minimum: `min = 0.0`. The conversion succeeded, and that is the problem. The
seven zero-scores passed straight through `to_numeric` because they are valid numbers.
`errors="coerce"` catches text that fails to parse; it cannot catch a valid number that
is not a real measurement. That check has to be written explicitly:

In [5]:
score_clean = score.replace(0, np.nan)  #np.nan replaces all 0s with NaN, import numpy as np (py package)

print(f"zeros converted to NaN: {(score == 0).sum()}")
print("min is now:", score_clean.min(), "| mean:", round(score_clean.mean(), 1))

zeros converted to NaN: 7
min is now: 81.5 | mean: 92.4


## Task 1.3: Exact duplicates

Some rows appear in the file twice, identical in every column:

In [7]:
n_dup = dining.duplicated().sum()
print(f"exact duplicate rows: {n_dup}")

deduped = dining.drop_duplicates()

# Print this line with every future labs/assignments
print(f"{len(dining)} rows in, {len(deduped)} out, {len(dining) - len(deduped)} dropped as exact duplicates")

exact duplicate rows: 46
366 rows in, 320 out, 46 dropped as exact duplicates


The last print line has the shape of a cleaning log entry: the action taken, the
number of rows affected, stated so a reader could re-derive it. Every cleaning action
for the rest of the course gets an entry in that form.

---
## Checkpoint: submit before leaving class

1. The two zeros: in which column is `0` legitimate, in which is it a disguised missing
   value, and what is the evidence?
2. How many exact duplicate rows are in the file, and what would keeping them do to a mean?
3. What should `-999` in `seats` become, and why?

*Answers here.*

In [ ]:
ct

1. The zeros contained in the 'seats' column are legitimate.
2. There are are total of 46 exact duplicate rows.
3. The value '-999' in seats is a sentinel and should be replaced with a NaN.
It should be replaced because it represents no real value, instead  of having an empty cell it should have a NaN over the sentinel value.


---
# Part 2: A3 (Cleaning I)

Graded. Four tasks. Task 2.4 carries the most rubric weight.

## Task 2.1: Sentinel policy, defended

Decide the fate of every sentinel found in `score` and `seats`: the zeros, the
`-999`s, the text values. For each: convert it, keep it, or flag it, with one sentence
of justification per decision. "Zero seats is real for food trucks" is a justification;
"converted everything to NaN" is not.

In [ ]:
# your conversions

*Decisions and justifications here.*

## Task 2.2: A ranking that reverses

Compute mean `score` by category twice, once treating the zero-scores as real values
and once as missing, and compare the rankings.

The categories first need consolidating (27 strings, ~6 real categories, profiled in
Lab 2). Building that mapping properly is Week 4's job, so this week it is provided for you:

In [ ]:
def canon(s):
    """Quick-and-dirty category canonicalizer. Week 4 teaches you to build this."""
    if pd.isna(s):
        return s
    s = " ".join(s.strip().lower().replace("-", " ").split())
    return {"c store": "convenience", "convenience store": "convenience",
            "coffee shop": "coffee", "café": "cafe",
            "fastcasual": "fast casual", "foodtruck": "food truck"}.get(s, s)

dining["category_clean"] = dining.category.map(canon)
print(dining.category_clean.value_counts())

In [ ]:
# your two group-bys: mean score by category_clean, with and without the zero-scores

Report the two means for the category that moves most, and both rankings. State which
category ranks worst under each treatment. Add two sentences on how a decision made in
Week 3 (sentinel policy) changes a conclusion that would be presented in Week 14.

## Task 2.3: Duplicates the naive check misses

`drop_duplicates()` with no arguments only catches rows that match exactly, byte for
byte. `"Port City Java "` and `"port city java"` are the same place with different case
and whitespace.

1. Normalize `location_name` (strip, lowercase, collapse internal whitespace) into a new
   column.
2. Count duplicates on `(unit_code, normalized name, inspection_date)`.
3. Report: exact duplicates, duplicates after normalizing, and how many the naive check
   missed. Show two rows that only the normalized check catches.

In [ ]:
# your near-duplicate hunt

## Task 2.4: Start the cleaning log

Assemble everything A3 did into a cleaning log: a markdown
table with one row per action.

| # | Action | Rows affected | Rows remaining | Why |
|---|---|---|---|---|

Requirements:

- **8 to 12 rows**, covering every change made (conversions, NaN decisions, drops)
- Every row carries a **count**, and the arithmetic must close: rows-in minus drops equals
  rows-out, exactly. Numbers that do not reconcile mean an action went unlogged; find it.
- The **Why** column is one specific clause. "Impossible value for this measurement"
  qualifies; "cleaning" does not.

This table is the seed of the P2 cleaning log (due Oct 1), where the same standard
applies to your own project data.

*Log here.*

---
## AI use note

List any AI tools used and what they were used for. If none, write "none." One or two
sentences.

*Answer here.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A3_[yourUnityID].ipynb`
4. Upload to the **A3** space on Moodle.

The **Checkpoint** section is submitted separately to **Week 3 In-Class Activity**, before
the end of class on Friday. Due for A3: **Thursday, Sep 10, 11:59 PM**.